# Using SQL in Python

## What is SQL?

- **SQL (Structured Query Language)**: A programming language designed for managing and manipulating relational databases. It allows you to create, read, update, and delete data in a database.
- SQL is used to communicate with databases and perform various operations such as querying data, inserting new records, updating existing records, and deleting records.
- SQL is a standard language for relational database management systems (RDBMS) and is widely used in various applications, including web development, data analysis, and data science.

## Relational Databases

- A **relational database** is a type of database that organizes data into tables (also called relations) consisting of rows and columns. Each table represents a specific entity, and the columns represent the attributes of that entity.

We will be using PostgreSQL, which is a popular open-source relational database management system (RDBMS). It is known for its robustness, scalability, and support for advanced features. PostgreSQL allows us to create and manage databases, define tables, and perform various operations using SQL.

In [81]:
# import the postgresql library for python

import psycopg2 # the PG library for Python
import os # for accessing environment variables (this keeps our password secure and out of the code)
import pandas as pd # for data manipulation and analysis (optional, but useful for working with query results)


## Connecting to PostgreSQL with Python

A RDBMS like PostgreSQL contains a server that manages the database and allows clients to connect and interact with it. To connect to a PostgreSQL database from Python, we can use the `psycopg2` library, which provides a convenient interface for working with PostgreSQL databases.

In [82]:
# Connect to the database
conn = psycopg2.connect(
    dbname="postgres", # or your database name
    user="postgres",  # or your database user
    password=os.environ.get("DB_PASSWORD"),  # set this in your environment
    host="localhost", # or your database host
    port="5433", # default PostgreSQL port
    options='-c client_encoding=UTF8'  # Force UTF-8 encoding at connection level
)

# Create a cursor object to interact with the database
cur = conn.cursor() # the cursor is like a pointer that allows us to execute SQL commands and fetch results

## Fetching Data from PostgreSQL

The postgresql server handles CRUD operations (Create, Read, Update, Delete). The server has databases, which have tables (or relations), which have rows (or records) and columns (or attributes). We use the SQL scripting language to interact with the database, and we can use Python to send SQL commands to the server and retrieve results.

In [83]:
# Example SQL query to select all records from a table called "artists"
get_artists_query = "SELECT * FROM artists;"

cur.execute(get_artists_query)

# Fetch all rows once (fetchall() can only be called once per query execution)
rows = cur.fetchall()
columns = [desc[0] for desc in cur.description]

# Configure pandas display settings for better readability
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.expand_frame_repr', False)

# Create and display the DataFrame
artists_df = pd.DataFrame(rows, columns=columns)
print("Artists Table:")
print(artists_df.to_string(index=False))

Artists Table:
 artist_id  artist_name       born      death
         1 Pablo Picaso 1881-10-25 1973-04-08
         2 Savador Dalí 1904-05-11 1989-01-23


Now let's look at the 'works' table in our database, which contains information about various works of art. The table has the following columns:

In [84]:
get_works_query = "SELECT * FROM works;"
cur.execute(get_works_query)
rows = cur.fetchall()
columns = [desc[0] for desc in cur.description]
works_df = pd.DataFrame(rows, columns=columns)

print("Works Table:")
print(works_df.to_string(index=False))

Works Table:
 work_id                                                                          work_name  year
       1                                                                           Guernica  1937
       2                                                          The Persistence of Memory  1931
       3 Dream Caused by the Flight of a Bee Around a Pomegranate a Second Before Awakening  1944
       4                                                          Les Demoiselles d’Avignon  1907


Now let's look at the join table:

In [85]:
get_join_query = "SELECT * FROM artists_works;"
cur.execute(get_join_query)
rows = cur.fetchall()
columns = [desc[0] for desc in cur.description]
join_df = pd.DataFrame(rows, columns=columns)

print("Artists-Works Join Table (maps which artist created which work):")
print(join_df)

Artists-Works Join Table (maps which artist created which work):
   artists_id  works_id
0         1.0       1.0
1         2.0       2.0
2         2.0       3.0
3         1.0       4.0
4         NaN       NaN


Now let's do the join:

In [86]:
# Join artists and works on artists_works
join_query = """
    SELECT 
        artists.artist_id,
        artists.artist_name,
        works.work_id,
        works.work_name,
        works.year
    FROM artists 
    JOIN artists_works ON artists.artist_id = artists_works.artists_id 
    JOIN works ON works.work_id = artists_works.works_id
    ORDER BY artists.artist_id, works.year;
"""
cur.execute(join_query)
rows = cur.fetchall()
columns = [desc[0] for desc in cur.description]
joined_df = pd.DataFrame(rows, columns=columns)

print("Complete Join: Artists and Their Works")
print(joined_df.to_string(index=False))

Complete Join: Artists and Their Works
 artist_id  artist_name  work_id                                                                          work_name  year
         1 Pablo Picaso        4                                                          Les Demoiselles d’Avignon  1907
         1 Pablo Picaso        1                                                                           Guernica  1937
         2 Savador Dalí        2                                                          The Persistence of Memory  1931
         2 Savador Dalí        3 Dream Caused by the Flight of a Bee Around a Pomegranate a Second Before Awakening  1944


## Updating Data in PostgreSQL

To update existing records in a PostgreSQL database using Python, we can use the `UPDATE` SQL command along with the `execute` method of the cursor object. After executing the update command, we need to commit the transaction to save the changes to the database.

**IMPORTANT**: Unlike last week when we used pandas to work with csv data and carried out operations in memory, **updating data in a database modifies the actual records**! Always **be cautious** when performing update operations, as they can modify or delete existing data. It's a good practice to **back up your database** before making significant changes. There is no "undo" button in SQL!

Ideally, you would back up your database with a weekly dump using the `pg_dump` command in your terminal. Pg_dump comes installed with PostgreSQL. Here is an example command to back up a PostgreSQL database:

```bash
pg_dump -U your_username -h your_host -p your_port -F c -b -v -f "your_backup_file.backup" your_database_name
```

These flags mean:

- `-U your_username`: Specifies the PostgreSQL username.
- `-h your_host`: Specifies the host where the PostgreSQL server is running.
- `-p your_port`: Specifies the port number on which the PostgreSQL server is listening
- `-F c`: Specifies the format of the backup file (custom format).
- `-b`: Includes large objects in the backup.
- `-v`: Enables verbose mode to display detailed information during the backup process.
- `-f "your_backup_file.backup"`: Specifies the name of the backup file

This script can be added to a **cron job** or scheduled task to automate regular backups. A 'cron job' is a scheduled task that runs automatically at specified intervals on Unix-like operating systems. For Windows, you can use the Task Scheduler to achieve similar functionality.


## INSERT into Table

If we have a new artist to add, we must use the insert command.

In [ ]:
# Let's insert a new artist called Diego Rivera, who was born in 1886 and died in 1957

insert_artist_query = """
INSERT INTO artists (artist_name, born, death) VALUES ('Diego Rivera', '1886-12-08', '1957-11-24');
"""
try:
	cur.execute(insert_artist_query)
	conn.commit()
except psycopg2.Error as e:
	conn.rollback()
	print(f"Insert failed, transaction rolled back: {e}")



In [ ]:
# Now Let's add a new work by Diego Rivera in the works table

insert_work_query = """
INSERT INTO works (work_name, year) VALUES ('Dream of a Sunday Afternoon in Alameda Central', 1947);
"""
try:
    cur.execute(insert_work_query)
    conn.commit()
except psycopg2.Error as e:
    conn.rollback()
    print(f"Insert failed, transaction rolled back: {e}")


In [ ]:
# Now let's manually link the new work to Diego Rivera in the artists_works join table (since we haven't set up triggers yet)

link_artist_work_query = """
INSERT INTO artists_works (artists_id, works_id)
SELECT a.artist_id, w.work_id
FROM artists a, works w
WHERE a.artist_name = 'Diego Rivera' AND w.work_name = 'Dream of a Sunday Afternoon in Alameda Central';
"""
try:
    cur.execute(link_artist_work_query)
    conn.commit()
except psycopg2.Error as e:
    conn.rollback()
    print(f"Insert failed, transaction rolled back: {e}")

In [100]:
# Check results of the join again to see if the new artist and work are linked correctly
cur.execute(join_query)
rows = cur.fetchall()
for row in rows:
    print(row)
    

(1, 'Pablo Picaso', 4, 'Les Demoiselles d’Avignon', 1907)
(1, 'Pablo Picaso', 1, 'Guernica', 1937)
(2, 'Savador Dalí', 2, 'The Persistence of Memory', 1931)
(2, 'Savador Dalí', 3, 'Dream Caused by the Flight of a Bee Around a Pomegranate a Second Before Awakening', 1944)
(3, 'Frida Kahlo', 5, 'The Two Fridas', 1939)
(3, 'Frida Kahlo', 6, 'Without Hope', 1945)


In [103]:
# Remove a work from the works table and also remove the corresponding record from the join table:
delete_work_query = """
DELETE FROM works WHERE work_name = 'Without Hope';
"""
try:
    cur.execute(delete_work_query)
    conn.commit()
except psycopg2.Error as e:
    conn.rollback()
    print(f"Delete failed, transaction rolled back: {e}")
    raise
# Check the join table again to see if the record linking "Without Hope" to its artist has been removed
cur.execute(join_query)
rows = cur.fetchall()
for row in rows:
    print(row)

(1, 'Pablo Picaso', 4, 'Les Demoiselles d’Avignon', 1907)
(1, 'Pablo Picaso', 1, 'Guernica', 1937)
(2, 'Savador Dalí', 3, 'Dream Caused by the Flight of a Bee Around a Pomegranate a Second Before Awakening', 1944)
(3, 'Frida Kahlo', 9, ' Self Portrait with Curly Hair', 1935)
(3, 'Frida Kahlo', 5, 'The Two Fridas', 1939)
(4, 'Leonardo da Vinci', 11, 'Mona Lisa', 1519)


## Flask and Forms

We can see that modifying data in a database is a powerful operation, but it is also quite cumbersome to do directly in SQL. In a real application, we would typically use a web framework like Flask to create a user-friendly interface for interacting with the database. Flask allows us to create forms that users can fill out to add new records or update existing records in the database without having to write SQL commands directly.

The steps to create a Flask application that interacts with a PostgreSQL database are as follows:
1. **Set up the Flask application**: Create a new Flask application and configure it to connect to your PostgreSQL database using the `psycopg2` library.
2. **Create routes and views**: Define routes in your Flask application that correspond to different actions (e.g., adding a new artist, updating an existing artist). Create views that render HTML templates with forms for user input.
3. **Handle form submissions**: In the view functions, handle form submissions by retrieving the data from the form, validating it, and then executing the appropriate SQL commands to insert or update records in the database.
4. **Run the Flask application**: Start the Flask development server and access the application through a web browser to interact with the database using the forms you created.
